# E23 — o operador: onde o critério pelos autovalores fica cego

O capítulo 18 mede um sistema de um número só: a estimativa de hoje é uma mistura da de ontem com
o dia de hoje, e a memória é 1/(1−a) dias. Ali o critério pelo autovalor é exato, e o §5 declara
para a pergunta E3 a referência a bater: **o critério pelos autovalores, que é cego fora do caso
simétrico**.

A família deste caderno tem o **mesmo autovalor** e acoplamentos diferentes:

    A(c) = [[0,9, c], [0, 0,9]]

O critério escalar devolve o mesmo número para toda coluna. A influência do estado inicial não é o
autovalor: é a norma da potência, ||A^k||, e ela **cresce antes de cair**.

In [1]:
# <- brinque com: RADIO, ACOPLAMENTOS, PASSOS, TOLERANCIA
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import graficos, operador

RAIZ = Path.cwd()
RADIO = operador.RADIO_PADRAO
ACOPLAMENTOS = operador.ACOPLAMENTOS_PADRAO
PASSOS = operador.PASSOS_PADRAO
TOLERANCIA = operador.TOLERANCIA_PADRAO

print("frevolab %s | raio espectral %.2f | acoplamentos %s | %d passos | tolerancia %.2f"
      % (frevolab.VERSAO, RADIO, ACOPLAMENTOS, PASSOS, TOLERANCIA))
print("o criterio escalar promete, para TODA a familia: memoria media %.1f dias | travessia %d dias"
      % (operador.memoria_do_autovalor(RADIO), operador.dias_do_autovalor(TOLERANCIA, RADIO)))

frevolab 0.1.0 | raio espectral 0.90 | acoplamentos (0.0, 1.0, 10.0, 100.0) | 200 passos | tolerancia 0.15
o criterio escalar promete, para TODA a familia: memoria media 10.0 dias | travessia 19 dias


In [2]:
# A familia: o que o autovalor promete e o que a norma entrega.
tabela = pd.DataFrame(operador.familia(ACOPLAMENTOS, PASSOS, TOLERANCIA, RADIO)).set_index("acoplamento")
print(tabela.round(3).to_string())
print()
print("o autovalor e o mesmo em todas as linhas: %s" % sorted({round(v, 6) for v in tabela["autovalor"]}))
print("e a norma entrega travessias que vao de %d a %d dias"
      % (tabela["memoria_norma"].min(), tabela["memoria_norma"].max()))

             autovalor  memoria_autovalor  pico_dia     pico  dias_autovalor  memoria_norma
acoplamento                                                                                
0.0                0.9               10.0         0    1.000              19             19
1.0                0.9               10.0         9    3.913              19             58
10.0               0.9               10.0         9   38.746              19             83
100.0              0.9               10.0         9  387.421              19            108

o autovalor e o mesmo em todas as linhas: [0.9]
e a norma entrega travessias que vao de 19 a 108 dias


In [3]:
# Figura 1: a influencia do estado inicial, dia a dia, para cada acoplamento.
fig, eixo = plt.subplots(figsize=(8.6, 4.4))
cores = ["#555555", "#1f4e79", "#2e7d32", "#b03a2e"]
for c, cor in zip(ACOPLAMENTOS, cores):
    serie = operador.normas(c, PASSOS, RADIO)
    eixo.semilogy(range(len(serie)), serie, color=cor, lw=1.6,
                  label="acoplamento %.0f (pico %.1f no dia %d)" % (c, serie.max(), int(serie.argmax())))
eixo.axhline(TOLERANCIA, color="#333333", ls=":", lw=1.2, label="a tolerância declarada")
eixo.axvline(operador.dias_do_autovalor(TOLERANCIA, RADIO), color="#333333", ls="--", lw=1.2,
             label="a travessia que o autovalor prevê: %d dias" % operador.dias_do_autovalor(TOLERANCIA, RADIO))
eixo.set_xlabel("dias, k")
eixo.set_ylabel("influência do estado inicial (norma da potência)")
eixo.legend(frameon=False, fontsize=8)
eixo.grid(alpha=0.25, which="both", ls=":")
fig.tight_layout()
graficos.salvar(fig, "E23_operador", 1)
plt.close(fig)
print("norma no caso diagonal, ultimos dias: %s" % np.round(operador.normas(0.0, 6, RADIO)[1:], 4))

norma no caso diagonal, ultimos dias: [0.9    0.81   0.729  0.6561 0.5905 0.5314]


## Leitura visual das figuras

Feita nesta sessão abrindo o .png com a ponte de visão (AGENTS.md §9), depois de o caderno rodar.

O que o desenho mostra, e o que só se vê olhando: o eixo horizontal vai de 0 a 200 dias e o
vertical é logarítmico, de 10^-9 a 10^3 --- nove décadas, de modo que uma queda exponencial sai
reta. A curva do acoplamento zero (cinza) é exatamente isso: uma reta que desce desde o dia zero.
As outras três **sobem primeiro**: a azul, a verde e a vermelha viram todas no nono dia, e viram
tanto mais alto quanto maior o acoplamento --- 3,9, 38,7 e 387,4 contra o valor de partida, que é
1 em todas. A linha pontilhada marca a tolerância e a tracejada vertical marca os 19 dias que o
autovalor prevê: a cinza cruza a tolerância em cima dessa linha, e a vermelha só muito depois
dela, perto do centésimo dia. Nenhuma curva cruza outra em ponto algum, e a legenda nomeia o pico
e o dia do pico de cada uma.

In [4]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
NOMES = {0.0: "zero", 1.0: "um", 10.0: "dez", 100.0: "cem"}
resultado = {
    "operador_raio": float(RADIO),
    "operador_acoplamentos": int(len(ACOPLAMENTOS)),
    "operador_passos": int(PASSOS),
    "operador_tolerancia": float(TOLERANCIA),
    "operador_memoria_media": float(operador.memoria_do_autovalor(RADIO)),
    "operador_dias_autovalor": int(operador.dias_do_autovalor(TOLERANCIA, RADIO)),
    "operador_dias_menor": int(tabela["memoria_norma"].min()),
    "operador_dias_maior": int(tabela["memoria_norma"].max()),
    "operador_taxa_maior": float(tabela["memoria_norma"].max() / tabela["memoria_norma"].min()),
}
for c in ACOPLAMENTOS:
    nome = NOMES[c]
    resultado["operador_pico_%s" % nome] = float(tabela.loc[c, "pico"])
    resultado["operador_pico_dia_%s" % nome] = int(tabela.loc[c, "pico_dia"])
    resultado["operador_dias_%s" % nome] = int(tabela.loc[c, "memoria_norma"])

caminho = Path("lab/resultados/E23_operador.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E23_operador.json gravado | 21 grandezas
